In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from optimization.portfolio import recommend
from explainability.rationale import contract_rationale, shape_and_basis_notes

zone = "nord"
archetype = "chemicals"
annual_kwh = 20_000_000
reference_zone = "sicilia"
rho = 8

result = recommend(zone, archetype, annual_kwh, rho, reference_zone=reference_zone)

In [2]:
for line in contract_rationale(result):
    print(line)

baseload (96%): costs €0.25M more on average than the cheapest option (vppa), but cuts CVaR by €0.18M, worth it at this risk setting.
vppa (4%): cheapest on average (€16.27M), but its CVaR (€16.79M) is €0.18M worse than baseload's tail.
Vs. staying on spot: saves €1.86M expected, cuts CVaR by €2.48M.


In [3]:
zone2, ref2 = "sicilia", "sardegna"
result2 = recommend(
    zone2, "chemicals", 50_000_000, rho=8,
    reference_zone=ref2, has_wholesale_market_access=True,
)
print(result2["weights"])
for line in contract_rationale(result2):
    print(line)

{'baseload': 1.0, 'pap_solar': 0.0, 'pap_wind': 0.0, 'vppa': 0.0, 'spot_only': 0.0}
baseload (100%): cheapest option and lowest tail risk, no tradeoff to make.
Vs. staying on spot: saves €1.85M expected, cuts CVaR by €2.41M.


In [4]:
for line in shape_and_basis_notes(result, zone, archetype, annual_kwh, reference_zone=reference_zone):
    print(line)

baseload: a fixed 2 MW block delivered flat across all 8,760 hours by construction it has zero weather driven shape mismatch (the residual vs. your load is a constant, not a weather year varying one), which is the direct reason its cost has the lowest variance of any contract type in this portfolio.
vppa: you buy 100% of load at nord's own price, but the CfD settles against sicilia's price, the gap (basis risk) averages €0.0/MWh with a scenario spread (std) of €0.4/MWh, on top of the shape mismatch from wind CF alone.
